---
title: "Extract population pharmacokinetic NLME studies"
author: 
  - name: Marian Klose
    orcid: 0009-0005-1706-6289
    email: marian.klose@fu-berlin.de
    affiliations:
      - name: Freie Universität Berlin
date: now
callout-appearance: minimal
execute:
  enabled: true
  echo: true
  message: true
  warning: true
format:
  html:
    toc: true
    number-sections: true
    embed-resources: true
---

This notebook extracts the abstract and the full body text of population pharmacokinetic NLME studies based on a saved query (.csv) to the PMC database.

## Preamble

In [1]:
# load packages
import pandas as pd
import os
import time
from metapub import PubMedFetcher
from Bio import Entrez
from bs4 import BeautifulSoup

C:\Users\mklose\AppData\Roaming\Python\Python312\site-packages\eutils\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Load PMC query results

The `pmc_query_results.csv` is based on a query to the PubMed Central (PMC) database (https://pmc.ncbi.nlm.nih.gov/search/?term=%27population+pharmacokinetics%27%5Btitle%5D) with the search term `"population pharmacokinetics"[title]`. It allows to save the results in a .csv file containing basic information, but lacking the abstract and full body text.

In [2]:
# read in pmc query results
df = pd.read_csv(
    "in/pmc_query_results.csv",
    dtype={"PMID": str}
)

# show head
df.head()

,PMID,Title,Authors,Citation,First Author,Journal/Book,Publication Year,Create Date,PMCID,NIHMS ID,DOI
0,40687901,Pharmacokinetics and Safety with Bioequivalenc...,"Wu B, Wang W, Zhang Q, Yu G, Lin J, Zhang T, Y...",Drug Des Devel Ther. 2025 Jul 14;19:6025-6035....,Wu B,Drug Des Devel Ther,2025,2025/07/21,PMC12274271,NaN,10.2147/DDDT.S508202
1,32034814,Evaluation of pharmacokinetics and safety with...,"Wang T, Wang Y, Lin S, Fang L, Lou S, Zhao D, ...",J Clin Lab Anal. 2020 Jun;34(6):e23228. doi: 1...,Wang T,J Clin Lab Anal,2020,2020/02/09,PMC7307347,NaN,10.1002/jcla.23228
2,30710324,Systemic Bioequivalence Is Unlikely to Equal T...,"Au JL, Lu Z, Abbiati RA, Wientjes MG.",AAPS J. 2019 Feb 1;21(2):24. doi: 10.1208/s122...,Au JL,AAPS J,2019,2019/02/03,PMC6432930,NIHMS1009981,10.1208/s12248-019-0296-z
3,40781573,Assessment of Pharmacokinetics and Safety with...,"Fu Q, Huang C, Yuan Y, Wang Y, Zhu B, Liu Y, T...",Drugs R D. 2025 Sep;25(3):263-274. doi: 10.100...,Fu Q,Drugs R D,2025,2025/08/09,PMC12460201,NaN,10.1007/s40268-025-00519-4
4,24151591,Pharmacokinetics and bioequivalence evaluation...,"Brioschi TM, Schramm SG, Kano EK, Koono EE, Ch...",Biomed Res Int. 2013;2013:281392. doi: 10.1155...,Brioschi TM,Biomed Res Int,2013,2013/10/24,PMC3787571,NaN,10.1155/2013/281392


## PubMed Setup

The API key `NCBI_API_KEY` is saved as an environment variable for increased request rates.

In [3]:
# define number of articles to process
num_of_articles = 200

# define pubmed email for pmc fetcher (Bio.Entrez)
Entrez.email = "marianklose96@gmail.com"
Entrez.api_key = os.getenv("NCBI_API_KEY")

# initialize PubMed fetcher (metapub)
fetch = PubMedFetcher()

## Query PubMed

For each article in the query results, the abstract and full body text is fetched from PubMed and saved in a new dataframe. The dataframe is then saved as a .csv file.

In [4]:
# initialize empty dict and list
df["Abstract"] = pd.NA
df["Body"] = pd.NA

# keep only first rows
df = df.head(num_of_articles)

# loop over articles and fetch abstract + body text
for idx, row in df.iterrows():
    # retrieve identifiers
    pmid = str(row["PMID"])
    pmcid = str(row["PMCID"])

    # fetch abstract from PubMed by pmid
    try:
        a = fetch.article_by_pmid(pmid)
        abstract = getattr(a, "abstract", None)
        df.at[idx, "Abstract"] = abstract if abstract else pd.NA
    except Exception as e:
        df.at[idx, "Abstract"] = pd.NA
        print(f"PMID {pmid}: abstract fetch failed: {e}")

    # don't overload NCBI servers
    time.sleep(0.5)

    # fetch full body text from PMC by PMCID
    try:
        # fetch full text xml from PMC
        xml = Entrez.efetch(db="pmc", id=pmcid, retmode="xml")
        soup = BeautifulSoup(xml, "lxml-xml")
        text = soup.get_text(separator=" ", strip=True)
        df.at[idx, "Body"] = text if text else pd.NA
    except Exception as e:
        df.at[idx, "Body"] = pd.NA
        print(f"PMCID {pmcid}: body fetch failed: {e}")

    # don't overload NCBI servers
    time.sleep(0.5) 

## Show Results

In [5]:
# show the dataframe
df

,PMID,Title,Authors,Citation,First Author,Journal/Book,Publication Year,Create Date,PMCID,NIHMS ID,DOI,Abstract,Body
0,40687901,Pharmacokinetics and Safety with Bioequivalenc...,"Wu B, Wang W, Zhang Q, Yu G, Lin J, Zhang T, Y...",Drug Des Devel Ther. 2025 Jul 14;19:6025-6035....,Wu B,Drug Des Devel Ther,2025,2025/07/21,PMC12274271,NaN,10.2147/DDDT.S508202,PURPOSE: Isosorbide mononitrate was recommende...,pmc Drug Des Devel Ther Drug Des Devel Ther 95...
1,32034814,Evaluation of pharmacokinetics and safety with...,"Wang T, Wang Y, Lin S, Fang L, Lou S, Zhao D, ...",J Clin Lab Anal. 2020 Jun;34(6):e23228. doi: 1...,Wang T,J Clin Lab Anal,2020,2020/02/09,PMC7307347,NaN,10.1002/jcla.23228,"BACKGROUND AND OBJECTIVE: Amlodipine, a main s...",J Clin Lab Anal J. Clin. Lab. Anal 3636 jclinl...
2,30710324,Systemic Bioequivalence Is Unlikely to Equal T...,"Au JL, Lu Z, Abbiati RA, Wientjes MG.",AAPS J. 2019 Feb 1;21(2):24. doi: 10.1208/s122...,Au JL,AAPS J,2019,2019/02/03,PMC6432930,NIHMS1009981,10.1208/s12248-019-0296-z,Approval of generic drugs by the US Food and D...,AAPS J AAPS J 319 nihpa The AAPS journal 1550-...
3,40781573,Assessment of Pharmacokinetics and Safety with...,"Fu Q, Huang C, Yuan Y, Wang Y, Zhu B, Liu Y, T...",Drugs R D. 2025 Sep;25(3):263-274. doi: 10.100...,Fu Q,Drugs R D,2025,2025/08/09,PMC12460201,NaN,10.1007/s40268-025-00519-4,"BACKGROUND AND OBJECTIVE: Nitroglycerin, a cor...",pmc Drugs R D Drugs R D 2559 drugsrd Drugs in ...
4,24151591,Pharmacokinetics and bioequivalence evaluation...,"Brioschi TM, Schramm SG, Kano EK, Koono EE, Ch...",Biomed Res Int. 2013;2013:281392. doi: 10.1155...,Brioschi TM,Biomed Res Int,2013,2013/10/24,PMC3787571,NaN,10.1155/2013/281392,The purpose of this study was to investigate c...,Biomed Res Int Biomed Res Int 2029 bmri BMRI B...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,32338459,Bioequivalence and pharmacodynamics of a gener...,"Li X, Liu L, Xu B, Xiang Q, Li Y, Zhang P, Wan...",Pharmacol Res Perspect. 2020 Apr;8(2):e00593. ...,Li X,Pharmacol Res Perspect,2020,2020/04/28,PMC7184321,NaN,10.1002/prp2.593,To assess bioequivalence of a generic dabigatr...,Pharmacol Res Perspect Pharmacol Res Perspect ...
196,18959779,Bioequivalence study of three ibuprofen formul...,"Bramlage P, Goldis A.",BMC Pharmacol. 2008 Oct 29;8:18. doi: 10.1186/...,Bramlage P,BMC Pharmacol,2008,2008/10/31,PMC2613135,NaN,10.1186/1471-2210-8-18,BACKGROUND: This phase I study was designed to...,BMC Pharmacol 56 bmcphar BMC Pharmacology 1471...
197,32099328,"Comparative Pharmacokinetics, Bioequivalence a...","Wang J, Yang T, Mei H, Yu X, Peng H, Wang R, C...",Drug Des Devel Ther. 2020 Jan 29;14:435-444. d...,Wang J,Drug Des Devel Ther,2020,2020/02/27,PMC6996485,NaN,10.2147/DDDT.S235064,OBJECTIVE: To evaluate the pharmacokinetics (P...,Drug Des Devel Ther Drug Des Devel Ther 958 dd...
198,21775984,Bioequivalence of oral products and the biopha...,"Amidon KS, Langguth P, Lennernäs H, Yu L, Amid...",Clin Pharmacol Ther. 2011 Sep;90(3):467-70. do...,Amidon KS,Clin Pharmacol Ther,2011,2011/07/22,PMC3228645,NIHMS337891,10.1038/clpt.2011.109,The demonstration of bioequivalence (BE) is an...,Clin Pharmacol Ther 319 nihpa Clinical pharmac...


## Save to JSON
JSON makes our life easier when we want to load the data again. Especially the body of the articles contains lots of commas, which would mess up a CSV file.

In [6]:
# save to json
df.to_json(
    "out/pmc_query_results_enriched.json",
    orient="records",
    indent=2,
    force_ascii=False
)